# Amazon ML Challenge — Business Entity Resolution: Colab training pipeline

Runs the full pipeline (normalize → block → features → train → predict) on Colab.

**Before running:** upload the `dataset/` folder (containing `train/` and `test/` TSVs) to your Google Drive, e.g. at
`MyDrive/amazon_ml_challenge/dataset/train/train_source1.tsv` etc. Update `DRIVE_DATASET_DIR` below if you used a different path.

A standard (CPU) Colab runtime is enough — LightGBM training here doesn't need a GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# --- Configuration ---
REPO_URL = "https://github.com/manasshete/amazon-ml-challenge.git"
REPO_DIR = "/content/amazon-ml-challenge"

# Where your uploaded dataset lives in Drive. Must contain train/ and test/ subfolders
# with the *_source1.tsv, *_source2.tsv, *_source3.tsv (and train_ground_truth.tsv) files.
DRIVE_DATASET_DIR = "/content/drive/MyDrive/amazon_ml_challenge/dataset"

# Where final models/outputs get copied back to Drive so they survive a runtime reset.
DRIVE_ARTIFACTS_DIR = "/content/drive/MyDrive/amazon_ml_challenge/artifacts"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/amazon_ml_challenge/output"

In [ ]:
import os

if os.path.isdir(REPO_DIR):
    !git -C "{REPO_DIR}" pull
else:
    !git clone "{REPO_URL}" "{REPO_DIR}"

PROJECT_DIR = os.path.join(
    REPO_DIR,
    "6ab10eb3b23ba_student_resource", "student_resource", "business_entity_resolution",
)
assert os.path.isdir(PROJECT_DIR), f"project dir not found: {PROJECT_DIR}"
print("Project dir:", PROJECT_DIR)

In [ ]:
!pip install -q -r "{PROJECT_DIR}/requirements.txt"

In [ ]:
# Link the Drive dataset into the exact relative path the scripts expect
# (business_entity_resolution/../dataset -> student_resource/dataset).
student_resource_dir = os.path.dirname(PROJECT_DIR)
repo_dataset_path = os.path.join(student_resource_dir, "dataset")

assert os.path.isdir(DRIVE_DATASET_DIR), (
    f"{DRIVE_DATASET_DIR} not found -- upload your dataset folder to Drive first "
    "(see the markdown cell above) or fix DRIVE_DATASET_DIR."
)

if os.path.islink(repo_dataset_path) or os.path.exists(repo_dataset_path):
    if os.path.islink(repo_dataset_path):
        os.remove(repo_dataset_path)
else:
    os.symlink(DRIVE_DATASET_DIR, repo_dataset_path)

print("dataset ->", os.path.realpath(repo_dataset_path))
!ls "{repo_dataset_path}/train" "{repo_dataset_path}/test"

In [ ]:
# Artifacts (normalized parquet cache, models) live on local Colab disk for speed --
# Drive I/O is slow for many small/medium files. We copy the important bits back
# to Drive at the end of the run so they survive a runtime disconnect.
local_artifacts = os.path.join(PROJECT_DIR, "artifacts")
os.makedirs(local_artifacts, exist_ok=True)
os.makedirs(DRIVE_ARTIFACTS_DIR, exist_ok=True)
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

## Run the pipeline

Each stage can be re-run independently once its inputs exist (e.g. re-run only `train` after
tweaking `src/train.py`, without re-normalizing/re-blocking).

In [ ]:
%cd {PROJECT_DIR}
!python -m src.run_pipeline --stage normalize

In [ ]:
!python -m src.run_pipeline --stage block

In [ ]:
!python -m src.run_pipeline --stage features

In [ ]:
!python -m src.run_pipeline --stage train

In [ ]:
!python -m src.run_pipeline --stage predict --out "{PROJECT_DIR}/../output"

In [ ]:
!python "{PROJECT_DIR}/../utils/validate_submission.py" \
  --matching "{PROJECT_DIR}/../output/matching_results.tsv" \
  --candidate "{PROJECT_DIR}/../output/candidate_pairs.tsv" \
  --test-dir "{repo_dataset_path}/test"

## Copy results back to Drive

In [ ]:
import shutil

# Only the small, important artifacts -- not the multi-GB normalized/feature caches.
keep = ["model_fold0.txt", "model_fold1.txt", "model_fold2.txt", "model_fold3.txt",
        "model_fold4.txt", "calibrator.pkl", "oof_predictions.parquet"]
for name in keep:
    src = os.path.join(local_artifacts, name)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(DRIVE_ARTIFACTS_DIR, name))

output_dir = os.path.join(student_resource_dir, "output")
for name in ["matching_results.tsv", "candidate_pairs.tsv"]:
    src = os.path.join(output_dir, name)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(DRIVE_OUTPUT_DIR, name))

print("Copied models + outputs to", DRIVE_ARTIFACTS_DIR, "and", DRIVE_OUTPUT_DIR)